# CymbalGoal—Intelligent Search: From Keywords to Hybrid

This notebook is your workbench for the whole lab.

Your AlloyDB **cluster** was built for you before you started. What it does *not* yet contain is a
database, a schema, any data, or any indexes. Task 1 is where you build all four—and that is
deliberate, not a shortcut. Creating the database yourself is what makes you its owner, and
`CREATE INDEX ... USING scann` is a genuinely interesting statement that deserves better than being
buried in a startup script nobody reads.

### How to use this notebook

- **Task 1** is these cells. Run them top to bottom.
- **Tasks 2–5** you write yourself, in new cells, using the `q()` helper this notebook defines.
  Every query lives in the lab instructions, so you can follow along later without this file.
- **Every cell here is safe to re-run.** Nothing drops your database, nothing reloads data that is
  already loaded. If something goes sideways, re-running is always a reasonable first move.

## Step 1—Install the client libraries

Two packages do the work:

- **`google-cloud-alloydb-connector`**—Google's AlloyDB connector. It handles the TLS handshake and
  the IAM token exchange, so you never build a connection string or hold a password.
- **`pg8000`**—a pure-Python PostgreSQL driver. The connector hands it an already-authenticated,
  already-encrypted socket.

This takes about thirty seconds, and it is the longest install in the lab.

In [ ]:
!pip install -q "google-cloud-alloydb-connector[pg8000]" pandas 2>&1 | tail -1
print("client libraries ready")

## Step 2—Discover your environment

Notice there is nothing to fill in below. No project ID, no region, no cluster name, no username.

Everything is discovered from the environment this runtime is already running in. That is what lets
the same notebook work for you, for the person beside you, and for you again in your own project
months from now—and it is why this file can live in a public GitHub repository without anyone
needing a credentials review.

**Read the code before you run it.** It is worth seeing how little is actually required.

In [ ]:
import subprocess, json, time, io, gzip, csv, textwrap

def sh(cmd):
    """Run a shell command and return its trimmed stdout."""
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()

PROJECT = sh("gcloud config get-value project")
USER    = sh("gcloud config get-value account")

# Find the cluster rather than assuming its name. A hardcoded name is a notebook
# that works exactly once, in exactly one project.
clusters = json.loads(sh("gcloud alloydb clusters list --format=json") or "[]")
assert clusters, "No AlloyDB cluster found in this project. Is the lab still provisioning?"
cluster  = clusters[0]
CLUSTER  = cluster["name"].split("/")[-1]
REGION   = cluster["name"].split("/locations/")[1].split("/")[0]

instances = json.loads(
    sh(f"gcloud alloydb instances list --cluster={CLUSTER} --region={REGION} --format=json") or "[]")
primary = [i for i in instances if i.get("instanceType") == "PRIMARY"]
assert primary, f"No PRIMARY instance found in cluster {CLUSTER}."
INSTANCE = primary[0]["name"].split("/")[-1]

INSTANCE_URI = f"projects/{PROJECT}/locations/{REGION}/clusters/{CLUSTER}/instances/{INSTANCE}"
DB_NAME = "cymbalgoal"
GCS     = "gs://class-demo/alloydb-labs/cymbalgoal"

print(f"project   {PROJECT}")
print(f"region    {REGION}")
print(f"cluster   {CLUSTER}")
print(f"instance  {INSTANCE}")
print(f"you       {USER}")
print(f"\ntarget    {INSTANCE_URI}")
print("\nNote what is absent from that output: a password.")

## Step 3—Connect, and build the helpers you will use all lab

`enable_iam_auth=True` is the whole trick. Instead of sending a username and password, the connector
exchanges your Google Cloud credentials for a short-lived token. AlloyDB verifies that token and looks
you up in its own list of IAM users. Your Google identity *is* your database login.

Three helpers get defined here. The one you will use most is **`q()`**:

| Helper | What it does |
| :---- | :---- |
| `q(sql)` | Run a query, return the results as a pandas DataFrame |
| `run(sql)` | Run a statement that returns no rows—DDL, `COPY`, `SET` |
| `explain(sql)` | Show the query plan. You will need this in Tasks 3 and 4 |

**`q()` is what Tasks 2 through 5 are built on.** When the lab hands you a query, add a new cell and
run it through `q()`.

In [ ]:
from google.cloud.alloydb.connector import Connector, IPTypes
import pandas as pd

connector = Connector()
_conn = None   # one session, reused across cells, reconnected if it drops

def _connect(db=DB_NAME):
    c = connector.connect(
        INSTANCE_URI, "pg8000",
        user=USER, db=db,
        enable_iam_auth=True,          # no password anywhere
        ip_type=IPTypes.PUBLIC,        # IAM gates access; the connector carries mTLS
    )
    c.autocommit = True
    return c

def _session():
    """Return a live connection, reconnecting if the previous one died."""
    global _conn
    if _conn is None:
        _conn = _connect()
        return _conn
    try:
        cur = _conn.cursor(); cur.execute("SELECT 1"); cur.close()
    except Exception:
        _conn = _connect()
    return _conn

def _notices(conn, show):
    # AlloyDB reports index-build statistics as PostgreSQL NOTICEs. pg8000 hands
    # them over as raw dicts with BYTE keys -- {b'S': b'NOTICE', b'M': b'...'} --
    # which is unreadable on a projector. Decode the message field.
    if not show:
        return
    for n in getattr(conn, "notices", []) or []:
        msg = n.get(b"M")
        if msg:
            print("   ", msg.decode())
    try:
        conn.notices.clear()
    except Exception:
        pass

def run(sql, db=None, show_notices=False):
    """Execute a statement that returns no rows."""
    conn = _connect(db) if db else _session()
    cur = conn.cursor()
    cur.execute(sql)
    cur.close()
    _notices(conn, show_notices)
    if db:
        conn.close()

def q(sql, show_notices=False):
    """Run a query and return a pandas DataFrame. This is your main tool."""
    conn = _session()
    cur = conn.cursor()
    cur.execute(sql)
    cols = [d[0] for d in cur.description] if cur.description else []
    rows = cur.fetchall() if cur.description else []
    cur.close()
    _notices(conn, show_notices)
    return pd.DataFrame(rows, columns=cols)

def explain(sql, analyze=True):
    """Print the query plan. Use this to prove an index is actually being used."""
    kw = "EXPLAIN (ANALYZE, BUFFERS)" if analyze else "EXPLAIN"
    conn = _session()
    cur = conn.cursor()
    cur.execute(f"{kw} {sql}")
    for row in cur.fetchall():
        print(row[0])
    cur.close()

# Connect to the default 'postgres' database first -- 'cymbalgoal' does not exist yet.
t0 = time.time()
probe = _connect("postgres")
cur = probe.cursor(); cur.execute("SELECT version()"); ver = cur.fetchone()[0]; cur.close()
probe.close()
print(f"connected in {time.time()-t0:.1f}s")
print(ver.split(" on ")[0])

## Step 4—Create your database

Your cluster arrived with only the default `postgres` database. You are about to create your own.

**Why you, and not the provisioning that built the cluster?** Because a database created by
automation is owned by `postgres`. AlloyDB's `alloydbsuperuser` role is deliberately *not* a true
PostgreSQL superuser, so you would not be able to drop or fully manage a database you supposedly
owned. Creating it yourself makes you the owner, with everything that implies.

The cell below is **guarded**: it checks whether the database already exists before creating it, and
it never drops anything. Re-running cells is normal, and no cell in this lab will punish you for it.

In [ ]:
conn = _connect("postgres")
cur = conn.cursor()
cur.execute("SELECT pg_catalog.pg_get_userbyid(datdba) FROM pg_database WHERE datname = %s", (DB_NAME,))
row = cur.fetchone()

if row is None:
    cur.execute(f'CREATE DATABASE "{DB_NAME}"')
    print(f"created database {DB_NAME}, owned by {USER}")
else:
    owner = row[0]
    if owner == USER:
        print(f"database {DB_NAME} already exists and you own it—nothing to do")
    else:
        raise SystemExit(
            f"database {DB_NAME} exists but is owned by '{owner}', not you ({USER}).\n"
            f"You would not be able to manage it. Have the owner drop it, then re-run this cell."
        )
cur.close(); conn.close()

## Step 5—Enable the extensions

PostgreSQL ships deliberately small, and capabilities arrive as extensions. Four matter here:

| Extension | What it gives you | First used in |
| :---- | :---- | :---- |
| `vector` | The `VECTOR` column type for storing embeddings | Task 1 |
| `alloydb_scann` | Google's ScaNN index for fast similarity search | Task 1 |
| `google_ml_integration` | Calling models from inside SQL—`ai.embedding()`, `ai.rank()` | Task 2 |
| `pg_textsearch` | BM25 relevance ranking for full-text search | Task 3 |

`pg_textsearch` and `alloydb_scann` both require the `alloydbsuperuser` role. You already hold it—provisioning granted it to your IAM identity, which is why the next cell simply works.

In [ ]:
for ext in ["vector", "alloydb_scann", "google_ml_integration", "pg_textsearch"]:
    run(f"CREATE EXTENSION IF NOT EXISTS {ext}")

display(q("""
    SELECT extname AS extension, extversion AS version
    FROM pg_extension
    WHERE extname IN ('vector','alloydb_scann','google_ml_integration','pg_textsearch')
    ORDER BY extname
"""))

## Step 6—Apply the schema

The schema is staged in Cloud Storage as `schema.sql`: eight tables, their constraints, and
thirty-eight `COMMENT ON` statements describing what each column means.

Those comments are not decoration. Point a language model at a schema like this one and ask it to
write SQL, and comments like these are a large part of what makes its answers correct—they record that
player names are *not* unique, and that a NULL `transfer_fee` means "unknown" while zero means "free."

**Notice what `schema.sql` does not contain: a single `CREATE INDEX`.** That is Step 11, and the
reason is worth waiting for.

In [ ]:
schema_sql = sh(f"gcloud storage cat {GCS}/schema.sql")
assert schema_sql.strip(), "could not read schema.sql from Cloud Storage"

print(f"schema.sql is {len(schema_sql):,} characters")
print(f"  CREATE TABLE statements: {schema_sql.upper().count('CREATE TABLE')}")
print(f"  COMMENT ON statements:   {schema_sql.upper().count('COMMENT ON')}")
print(f"  CREATE INDEX statements: {schema_sql.upper().count('CREATE INDEX')}   <- deliberately zero")

# The table this whole lab revolves around, in full.
start = schema_sql.upper().find("CREATE TABLE PLAYERS")
end   = schema_sql.find("\n);", start)
print("\n" + "=" * 78)
print(schema_sql[start:end + 3])

# Three comments that carry real weight. These are not documentation for humans.
print("=" * 78)
print("A few of the 38 column comments:\n")
for line in schema_sql.splitlines():
    s = line.strip()
    if s.upper().startswith("COMMENT ON") and any(
            k in s for k in ("player_name", "transfer_fee", "to_club_id")):
        print(textwrap.fill(s, 78, subsequent_indent="    "), "\n")

In [ ]:
# schema_sql was fetched and displayed in the cell above.
EXPECTED = ["competitions", "clubs", "players", "games",
            "appearances", "game_events", "player_valuations", "transfers"]

present = q(f"""
    SELECT count(*) AS n FROM information_schema.tables
    WHERE table_schema = 'public' AND table_type = 'BASE TABLE'
      AND table_name IN ({','.join("'" + t + "'" for t in EXPECTED)})
""")["n"][0]

if present == len(EXPECTED):
    print(f"\nall {present} tables already exist—skipping (schema.sql is not re-runnable)")
elif present == 0:
    t0 = time.time()
    run(schema_sql)
    print(f"\nschema applied in {time.time()-t0:.1f}s")
else:
    raise SystemExit(
        f"\n{present} of {len(EXPECTED)} tables exist. That is a half-built schema.\n"
        f"Drop the ones that exist, or drop and recreate the database, then re-run."
    )

display(q("""
    SELECT table_name,
           (SELECT count(*) FROM information_schema.columns c
             WHERE c.table_name = t.table_name AND c.table_schema = 'public') AS columns
    FROM information_schema.tables t
    WHERE table_schema = 'public' AND table_type = 'BASE TABLE'
      AND left(table_name, 1) <> '_'
    ORDER BY table_name
"""))

## Step 7—Preflight the load

Before moving a single byte, this cell compares the number of fields in each staged file against the
column list we intend to load it into, and refuses to continue on a mismatch.

**This guard is scar tissue, not paranoia.** A column list *longer* than the file fails loudly and
harmlessly. A list of the *same length in a different order* loads silently—putting stadium capacity
into market value—and the first symptom is a query returning nonsense in front of a room. Cheap
check, expensive alternative.

One subtlety it handles: the manifest's `column_order` is derived from the table definition, so for
`players` and `clubs` it includes `profile_text` and `profile_embedding`—columns the Step 8 files do
not carry, because those arrive separately in Step 9. Those get subtracted here.

In [ ]:
PASS2 = {"profile_text", "profile_embedding"}
ORDER = ["competitions", "clubs", "players", "games",
         "appearances", "game_events", "player_valuations", "transfers"]

manifest = json.loads(sh(f"gcloud storage cat {GCS}/manifest.json"))
staged   = manifest.get("staged_files")
items    = staged.items() if isinstance(staged, dict) else [(f.get("name"), f) for f in staged]

COLS = {}
for key, meta in items:
    if isinstance(meta, dict) and meta.get("column_order"):
        table = str(key).split("/")[-1].replace(".csv.gz", "").replace(".csv", "")
        COLS[table] = [c for c in meta["column_order"] if c not in PASS2]

for t in ORDER:
    assert t in COLS, f"{t}: no column_order in the manifest—never guess at this"
    proc = subprocess.Popen(["gcloud", "storage", "cat", f"{GCS}/{t}.csv.gz"],
                            stdout=subprocess.PIPE)
    with gzip.GzipFile(fileobj=proc.stdout, mode="rb") as gz:
        first = next(csv.reader(io.TextIOWrapper(gz, encoding="utf-8")))
    proc.stdout.close(); proc.wait()
    assert len(first) == len(COLS[t]), (
        f"{t}: file has {len(first)} fields, column list has {len(COLS[t])}. Refusing to load.")
    print(f"  {t:20s} {len(first):>3} fields  OK")

print("\npreflight passed—every file matches its column list")

## Step 8—Load the eight relational tables

This is a client-side `COPY`: data streams from Cloud Storage, through this runtime, into AlloyDB.

`COPY` is PostgreSQL's bulk path, and it is dramatically faster than row-by-row `INSERT` because the
server parses a stream instead of planning and executing millions of individual statements. If you
take one habit home from this step, make it that one—reaching for `INSERT` in a loop is the single
most common reason a data load takes hours instead of minutes.

`appearances` is the big one at 832,193 rows and will dominate the time here. Expect a couple of
minutes for the whole step.

Note the explicit column list on every `COPY`. Positional loading works right up until someone adds a
column, at which point every field silently shifts by one.

In [ ]:
def copy_table(table, columns, gcs_path):
    """Stream a gzipped CSV from Cloud Storage straight into COPY."""
    conn = _session()
    cur  = conn.cursor()
    proc = subprocess.Popen(["gcloud", "storage", "cat", gcs_path], stdout=subprocess.PIPE)
    stream = gzip.GzipFile(fileobj=proc.stdout, mode="rb")
    collist = ", ".join(f'"{c}"' for c in columns)
    cur.execute(f'COPY {table} ({collist}) FROM STDIN WITH (FORMAT csv)', stream=stream)
    cur.close()
    proc.stdout.close(); proc.wait()

total0 = time.time()
for t in ORDER:
    existing = q(f"SELECT count(*) AS n FROM {t}")["n"][0]
    if existing:
        print(f"  {t:20s} {existing:>9,} rows already present—skipping")
        continue
    t0 = time.time()
    copy_table(t, COLS[t], f"{GCS}/{t}.csv.gz")
    n = q(f"SELECT count(*) AS n FROM {t}")["n"][0]
    print(f"  {t:20s} {n:>9,} rows  {time.time()-t0:>6.1f}s")

print(f"\npass 1 complete in {time.time()-total0:.1f}s")

## Step 9—Load the profiles and their embeddings

Now the part that makes this lab possible.

Transfermarkt gives you numbers and categories: excellent for SQL, useless for searching by meaning.
So every player and club here also carries a **scouting profile**—roughly 250 words of narrative
prose describing how that player actually plays. Alongside it sits a **3,072-dimension embedding** of
that prose.

**All of it was generated once, offline, and staged.** Not to save you effort, but to make the lab
honest. Generating 14,235 grounded profiles costs real money and hours of wall clock, and generative
output drifts between runs. Pre-building means every student searches a byte-identical corpus—so
when the lab claims a particular query returns Neymar first, it does.

These land in staging tables and then `UPDATE` into place, because profiles are maintained separately
from the relational data and either can be regenerated without reloading the other.

In [ ]:
for t, key in [("players", "player_id"), ("clubs", "club_id")]:
    run(f"""CREATE TABLE IF NOT EXISTS _{t}_profiles (
                {key} INTEGER, profile_text TEXT, profile_embedding VECTOR(3072))""")
    if q(f"SELECT count(*) AS n FROM _{t}_profiles")["n"][0] == 0:
        t0 = time.time()
        copy_table(f"_{t}_profiles", [key, "profile_text", "profile_embedding"],
                   f"{GCS}/{t}_profiles.csv.gz")
        print(f"  staged {t} profiles in {time.time()-t0:.1f}s")

    run(f"""UPDATE {t} tgt
               SET profile_text      = src.profile_text,
                   profile_embedding = src.profile_embedding
              FROM _{t}_profiles src
             WHERE tgt.{key} = src.{key}""")

# Assert BOTH tables. A silent zero-row load is exactly the kind of failure that
# survives a happy-path check and then breaks Task 2 in front of a room.
n_players = q("SELECT count(*) AS n FROM players WHERE profile_embedding IS NOT NULL")["n"][0]
n_clubs   = q("SELECT count(*) AS n FROM clubs   WHERE profile_embedding IS NOT NULL")["n"][0]
assert n_players == 13439, f"expected 13,439 player profiles, got {n_players:,}"
assert n_clubs   == 796,   f"expected 796 club profiles, got {n_clubs:,}"
print(f"\n  {n_players:,} player profiles and {n_clubs:,} club profiles loaded and verified")

## Step 10—Look at what you just loaded

Before indexing any of it, spend thirty seconds on a single row. If "vector embedding" is a phrase
you have nodded along to without ever quite pinning down, this is the cell that fixes that.

In [ ]:
row = q("""
    SELECT player_name, main_position, profile_text,
           vector_dims(profile_embedding) AS dimensions,
           profile_embedding::text        AS full_vector
    FROM players
    WHERE player_name = 'Mauro Icardi' AND profile_embedding IS NOT NULL
    LIMIT 1
""")

r = row.iloc[0]
print(f"{r['player_name']}  ({r['main_position']})\n")
print("PROFILE TEXT—what a scout would write")
print("-" * 78)
print(textwrap.fill(r["profile_text"][:600].rsplit(" ", 1)[0] + " ...", 78))
print()
print(f"PROFILE EMBEDDING—the same meaning, as {r['dimensions']:,} numbers")
print("-" * 78)
head = r["full_vector"].strip("[]").split(",")[:8]
print("[" + ", ".join(f"{float(v):+.5f}" for v in head) + f", ... {r['dimensions'] - 8:,} more ]")

### So what is an embedding?

Those 3,072 numbers are a *position*. A model read the scouting report and placed it at one specific
point in a space with 3,072 axes, arranged so that **distance means similarity**.

That single property buys you everything else in this lab. Two players whose profiles share not one
word in common can still sit close together, because they were described as doing the same job.
Search stops being "which documents contain these characters" and becomes "which documents are *near*
this idea."

Three things worth holding onto:

- **The individual numbers mean nothing on their own.** No axis is "finishing ability." Only relative
  positions carry meaning, so resist the urge to interpret a single dimension.
- **3,072 is not a free choice here.** In Task 2 you will embed *your* search text from inside SQL, and
  `ai.embedding()` returns whatever width the model natively produces. Stored vectors of any other
  width would fail the comparison outright.
- **It is not magic, and you will watch it fail.** Ask this corpus for *"someone who can unlock a
  parked bus"* and vector search confidently returns a **goalkeeper**. Hold on to that when Task 4
  argues for combining methods rather than crowning one.

## Step 11—Build the indexes, now that the data has arrived

Order matters here, and it is the most portable lesson in Task 1.

**Build an index first, and every row you load afterwards has to be inserted into it one at a time.**
Load first, and the index is built once, in bulk, from data it can see in its entirety. That rule
holds for any bulk load into any indexed table you will ever own.

For ScaNN it is not merely faster, it is *required*. ScaNN works by clustering your vectors into
partitions and searching only the promising ones. On an empty table there is nothing to cluster, and
AlloyDB refuses outright:

```
FAILED_PRECONDITION: Cannot create ScaNN index with empty table "players"
```

The index definitions live in a staged file rather than being retyped here, so there is exactly one
authoritative copy of them. **Read them before you run them**—those ScaNN statements are the most
interesting DDL in this lab.

In [ ]:
indexes_sql = sh(f"gcloud storage cat {GCS}/indexes.sql")
assert indexes_sql.strip(), "could not read indexes.sql from Cloud Storage"
print(indexes_sql)

Reading that ScaNN statement:

- **`USING scann (profile_embedding cosine)`**—index this column, measuring distance by the *angle*
  between vectors rather than their length. For text embeddings, direction carries the meaning.
- **`num_leaves = 115`**—how many partitions to cluster 13,439 vectors into. Roughly the square root
  of the row count, which balances "how many partitions must I search" against "how many vectors are
  in each one."
- **`quantizer = 'sq8'`**—store each dimension compressed to eight bits. A little precision traded
  for a much smaller index that comfortably stays in memory.

The six `btree` indexes are ordinary PostgreSQL, on foreign key columns. Worth knowing:
**PostgreSQL does not index foreign keys automatically.** It indexes primary keys and unique
constraints; the child side of every relationship is yours to handle.

`maintenance_work_mem` gets raised first, because index builds use it as working space and the default
is far too small for this.

In [ ]:
already = q("""
    SELECT count(*) AS n FROM pg_indexes
    WHERE schemaname = 'public' AND indexname LIKE '%scann%'
""")["n"][0]

if already >= 2:
    print("indexes already built—skipping (indexes.sql is not re-runnable)")
else:
    t0 = time.time()
    run("SET maintenance_work_mem = '2GB'")
    run(indexes_sql, show_notices=True)
    print(f"indexes built in {time.time()-t0:.1f}s")

display(q("""
    SELECT tablename, indexname
    FROM pg_indexes
    WHERE schemaname = 'public' AND tablename IN ('players','clubs')
    ORDER BY tablename, indexname
"""))

## Step 12—Verify, and record what you built

A quick census. These numbers are fixed: the corpus is a pinned snapshot, so everyone in the room sees
exactly these values, and so will you if you come back to this in two weeks.

In [ ]:
display(q("""
    SELECT 'players' AS table_name, count(*) AS rows,
           count(profile_embedding) AS with_embeddings FROM players
    UNION ALL SELECT 'clubs',       count(*), count(profile_embedding) FROM clubs
    UNION ALL SELECT 'games',       count(*), NULL FROM games
    UNION ALL SELECT 'appearances', count(*), NULL FROM appearances
    UNION ALL SELECT 'game_events', count(*), NULL FROM game_events
    UNION ALL SELECT 'transfers',   count(*), NULL FROM transfers
"""))

run("""CREATE TABLE IF NOT EXISTS provisioning_status (
         finished_at timestamptz PRIMARY KEY DEFAULT now(),
         players bigint, clubs bigint, appearances bigint)""")
run("""INSERT INTO provisioning_status (players, clubs, appearances)
       SELECT (SELECT count(*) FROM players WHERE profile_embedding IS NOT NULL),
              (SELECT count(*) FROM clubs   WHERE profile_embedding IS NOT NULL),
              (SELECT count(*) FROM appearances)""")

print("\nExpected: 13,439 players / 796 clubs / 832,193 appearances")
print("Task 1 complete. Your database is built, loaded, and indexed.")

## From here on, you write the queries

Tasks 2 through 5 are yours. The lab instructions give you each query; add a **new code cell** and run
it through `q()`, like this:

```python
q("""
    SELECT player_name, main_position
    FROM players
    WHERE profile_text ILIKE '%centre-back%'
    LIMIT 5
""")
```

Two habits worth adopting right now:

- **Change the search text and re-run.** The interesting part of Tasks 2 through 4 is never a single
  result. It is what happens to the ranking when you phrase the same need differently.
- **Reach for `explain()` when a query feels slow, or suspiciously fast.** In Tasks 3 and 4 you will
  use it to prove that the index you built is the one doing the work. It is easy to write a query that
  returns the right answer while quietly ignoring your index entirely.

The cell below is a working example. Run it to confirm `q()` behaves, then head back to the lab
instructions for Task 2.

In [ ]:
q("""
    SELECT player_name, main_position, country_of_citizenship
    FROM players
    WHERE profile_embedding IS NOT NULL
    ORDER BY player_id
    LIMIT 5
""")